# Hebrew Acronym Disambiguation: Experimental Study

Compare **Qwen free expansion with the sentence**, **the same Qwen version selecting from candidates with the sentence**, and **Ben’s trained DictaBERT selecting from that inventory** on the same identified dev items. This comparison does not estimate the contribution of context.

The default is a safe preview with two invented items: no research reads, model loading, requests, training or saved outputs. The prediction paths are connected; fixture checks are engineering evidence, not research results. All responses remain **unscored** pending the shared parsing and judgment protocol.

## 1. Setup and explicit controls

Use an installed checkout of this branch in the Colab kernel (`pip install -e /content/hebrew-acronym-disambiguation` after placing the checkout there), then restart the kernel if needed. Installation is a separate user action; this notebook does not install dependencies or download weights. Set `ROOT` to that checkout if the working directory is elsewhere. Jupyter/IPython are supplied by Colab.

For a new run, set `MODE="run"`, fill paths and exact Qwen version, choose the encoder route, and enable the requested calls. For inspection only, set `MODE="reload"`, `SAVED_RUN` and `SAVED_RUN_ID`; the artifact contains its input snapshot and all three conditions, so no original data/checkpoint/service is needed. Restart and execute top to bottom after changing controls.

In [ ]:
import gc
from pathlib import Path
from uuid import uuid4
from IPython.display import HTML, display

from hebrew_acronyms import experimental_study as study
from hebrew_acronyms.models.common.pairs import load_rows
from hebrew_acronyms.models.dictabert_cross_encoder.workflow import enable_offline
from hebrew_acronyms.models.dictabert_cross_encoder.model import load_finetuned, file_digest
from hebrew_acronyms.models.dictabert_cross_encoder.eval import evaluate as encoder_evaluate
from hebrew_acronyms.models.qwen.eval import ollama_generate, OLLAMA_URL

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
MODE = "preview"  # preview | run | reload
RUNTIME = "Colab"
SOURCE_REVISION = None  # Commit SHA if using an archive without .git; source hashes are always saved
INPUT_PATH = ROOT / "data/study_v1/encoder_inputs/dev.csv"
EXPECTED_DEV_ITEMS = 62
CHECKPOINT = None  # Path to Ben's weights; adjacent <weights>.json is mandatory
SAVED_ENCODER = None  # study.json from an identified encoder run, instead of CHECKPOINT
SAVED_ENCODER_RUN_ID = None
ENABLE_ENCODER = False
ENABLE_LLM = False
QWEN_MODEL = None  # Required exact Ollama model tag; no model is selected implicitly
QWEN_REVISION = None  # Required resolved immutable version/digest, supplied by the operator
DEVICE = "cuda"  # Explicitly choose "cpu" if required by your runtime
RUN_ID = str(uuid4())
OUTPUT_DIR = Path.home() / "nlp-study-runs" / RUN_ID  # New directory outside every Git checkout
SAVED_RUN = None
SAVED_RUN_ID = None

## 2. Local readiness

This checks interfaces and paths without loading a model or contacting a service. It does not certify checkpoint or provider compatibility. Qwen’s existing backend calls **Ollama at `http://localhost:11434` inside the Colab runtime**. The user must provision the exact model and service separately; the notebook neither starts Ollama nor pulls weights. A computer’s own localhost is not Colab’s localhost. The resolved Qwen digest is operator-supplied provenance; the current backend does not verify or pin it. Keep that deployment unchanged between the two modes.

Ben must supply the weights, their matching JSON, the exact base-model/tokenizer snapshot at the recorded location, and a compatible library environment. `load_finetuned` enforces all existing identity checks. Moving a checkpoint to Colab can fail those checks; request compatible artifacts/environment from Ben rather than editing the JSON or overriding identity checks.

In [ ]:
if MODE not in {"preview", "run", "reload"}:
    raise ValueError("MODE must be preview, run or reload")
if MODE == "reload":
    if SAVED_RUN is None or not SAVED_RUN_ID:
        raise ValueError("Set SAVED_RUN and its exact SAVED_RUN_ID")
    readiness = {"mode": "reload", "model_calls": False}
else:
    readiness = study.check_readiness(
        ROOT, INPUT_PATH, CHECKPOINT, OUTPUT_DIR, run=MODE == "run",
        encoder=ENABLE_ENCODER, llm=ENABLE_LLM,
        model=QWEN_MODEL, revision=QWEN_REVISION, saved_encoder=SAVED_ENCODER,
    )
    if MODE == "run" and SAVED_ENCODER is not None and not SAVED_ENCODER_RUN_ID:
        raise ValueError("Set the exact source run ID for saved encoder predictions")
print(readiness)

## 3. Identified input and a fresh output

The run route reads only the qualified `encoder_inputs/dev.csv` (expected 62 items); it never falls back to historical dev, test or mixed review files. The input identity binds the complete rows, their order and IDs. A new artifact starts with one `not_run` record for every item and condition, then is saved before inference. This keeps untouched or failed attempts visible.

Artifacts store input rows, settings, source hashes, raw prompts/responses and exact candidate orders. Keep the output directory outside Git. Each rerun uses a new run ID and directory; reload is read-only.

In [ ]:
if MODE == "reload":
    artifact = study.load_artifact(SAVED_RUN, expected_run_id=SAVED_RUN_ID)
    rows = artifact["items"]
else:
    rows = load_rows(INPUT_PATH) if MODE == "run" else study.fixture_rows()
    if MODE == "run" and len(rows) != EXPECTED_DEV_ITEMS:
        raise ValueError("Qualified dev item count changed; verify the selected input before running")
    settings = {
        "runtime": RUNTIME, "input_path": str(INPUT_PATH) if MODE == "run" else "invented_fixture",
        "input_kind": "qualified_dev" if MODE == "run" else "invented_fixture",
        "device": DEVICE, "qwen_model": QWEN_MODEL, "qwen_revision": QWEN_REVISION,
        "qwen_revision_verification": "operator-supplied; backend does not verify",
        "qwen_endpoint": OLLAMA_URL, "qwen_request_settings": "backend defaults; stream=False",
        "llm_modes": ["generate", "select"], "sentence_included": True,
        "selection_order": "shared SHUFFLE_SEED; one RNG over ordered valid selection rows",
        "selection_shuffle_seed": study.prompts.SHUFFLE_SEED,
        "encoder_enabled": ENABLE_ENCODER, "llm_enabled": ENABLE_LLM,
        "checkpoint": str(CHECKPOINT) if CHECKPOINT is not None else None,
        "saved_encoder": str(SAVED_ENCODER) if SAVED_ENCODER is not None else None,
        "saved_encoder_run_id": SAVED_ENCODER_RUN_ID,
        "scoring": "unscored; shared research parsing/judgment/aggregation pending",
    }
    provenance = study.source_provenance(ROOT, SOURCE_REVISION)
    artifact = study.new_artifact(rows, RUN_ID, settings, provenance)
    if MODE == "run":
        artifact_path = study.save_artifact(artifact, OUTPUT_DIR, create=True)
        print("Run:", RUN_ID, "Artifact:", artifact_path)
print("Items:", len(rows), "Input SHA-256:", artifact["input_identity"]["sha256"])

## 4. DictaBERT: checkpoint or saved predictions

Checkpoint inference calls the existing loader and evaluator directly. No training is included. The evaluator retains input, encoding and model failures. Loading failures stop here with the original error and preserve the initial artifact. Checkpoint training identities describe Ben’s training/dev inputs; they are not silently replaced by this prediction input.

Saved predictions must be an identity-bound `study.json`, with the exact source run ID and identical full input rows. Bare historical CSVs or evaluator lists cannot prove their input/run origin and are rejected. Ben can use this notebook with `ENABLE_ENCODER=True`, `ENABLE_LLM=False` to create that file. For an already retained evaluator list, use `new_artifact(original_rows, original_run_id, settings, provenance)`, then `attach_encoder(..., origin=...)` and `save_artifact(..., create=True)` **at its source**, supplying the actual checkpoint/hash and original provenance; do not reconstruct missing provenance by guessing.

DictaBERT is released before Qwen starts. When both use the same GPU, the operator must also ensure Ollama has no model resident before the encoder step.

In [ ]:
encoder_predictions = None
if MODE == "run" and (ENABLE_ENCODER or SAVED_ENCODER is not None):
    study.validate_artifact(artifact, expected_run_id=RUN_ID, rows=rows)
    if any(r["status"] != "not_run" for r in artifact["records"] if r["condition"] == "dictabert"):
        raise ValueError("Encoder predictions already exist; start a new run")
    current_encoder_settings = (str(CHECKPOINT) if CHECKPOINT is not None else None, DEVICE,
                               str(SAVED_ENCODER) if SAVED_ENCODER is not None else None, SAVED_ENCODER_RUN_ID)
    recorded_encoder_settings = tuple(artifact["settings"][key] for key in
                                      ("checkpoint", "device", "saved_encoder", "saved_encoder_run_id"))
    if current_encoder_settings != recorded_encoder_settings:
        raise ValueError("Encoder settings changed; restart with a new run")
if MODE == "run" and SAVED_ENCODER is not None:
    encoder_predictions, encoder_origin = study.saved_encoder_predictions(
        SAVED_ENCODER, expected_run_id=SAVED_ENCODER_RUN_ID, rows=rows,
    )
    study.attach_encoder(artifact, encoder_predictions, origin=encoder_origin)
    study.save_artifact(artifact, OUTPUT_DIR)
elif MODE == "run" and ENABLE_ENCODER:
    enable_offline()  # Local snapshot only; never download missing model files
    tokenizer, encoder, open_id, close_id = load_finetuned(str(CHECKPOINT), device=DEVICE)
    try:
        encoder_predictions = encoder_evaluate(rows, tokenizer, encoder, open_id, close_id, DEVICE)
        encoder_origin = {
            "checkpoint": str(Path(CHECKPOINT).resolve()),
            "weights_sha256": file_digest(CHECKPOINT),
            "manifest_sha256": file_digest(str(CHECKPOINT) + ".json"),
            "checkpoint_metadata": encoder.checkpoint_metadata,
        }
        study.attach_encoder(artifact, encoder_predictions, origin=encoder_origin)
        study.save_artifact(artifact, OUTPUT_DIR)
    finally:
        del encoder, tokenizer
        gc.collect()
        if DEVICE.startswith("cuda"):
            import torch
            torch.cuda.empty_cache()
print("Encoder statuses:", [r["status"] for r in artifact["records"] if r["condition"] == "dictabert"])

## 5. Qwen generation with the sentence

Use the existing source prompt with `target_raw` (including its prefix) and the unchanged sentence. This mode sends no candidate inventory or gold answer. Successful responses remain raw and unscored. Service failures retain their item records; each attempt is saved. No automatic retry or research scoring is added.

In [ ]:
if MODE == "run" and ENABLE_LLM:
    generate_response = lambda prompt, model=artifact["settings"]["qwen_model"]: ollama_generate(prompt, model=model)
    persist_attempt = lambda current: study.save_artifact(current, OUTPUT_DIR)
    generation_results = study.collect_responses(
        artifact, "generate", generate_response, on_record=persist_attempt,
    )
print("Generation statuses:", [r["status"] for r in artifact["records"] if r["condition"] == "generate"])

## 6. The same Qwen selecting with the sentence

The exact same callable/model/version receives the same sentence and raw target, plus candidates. The historical shared shuffle seed supplies a reproducible displayed order, which is saved per item; this is an implementation setting, not an approved scientific ordering policy. Singletons are retained. More than 26 candidates or duplicate entries produce explicit failures because the shared letter prompt cannot represent them safely.

The existing first-letter parser and substring-based generation scores are historical. They are **not called** here. Selection responses also remain unparsed/unscored until the shared owner supplies the agreed parser. The existing Qwen backend has no request timeout; if a request stalls, interrupt the cell and inspect the artifact before starting a new run.

In [ ]:
if MODE == "run" and ENABLE_LLM:
    selection_results = study.collect_responses(
        artifact, "select", generate_response, on_record=persist_attempt,
    )
print("Selection statuses:", [r["status"] for r in artifact["records"] if r["condition"] == "select"])

## 7. Inspect the same identified items

The table shows predictions, raw responses and failures across the three conditions. Counts below are operational statuses over all requested items; they are not accuracy or a research denominator policy. Failed and unrun items remain present. Reload validates item, run and model identities before display and does not combine historical output files.

To finish research scoring, the shared evaluation owner must provide approved selection parsing, generation judgments, and aggregation/denominator rules that retain failures across these same IDs. Until then no accuracy, validity rate or performance comparison is reported.

In [ ]:
study.validate_artifact(artifact, expected_run_id=artifact["run_id"])
prediction_table, status_counts = study.display_tables(artifact)
display(HTML(prediction_table))
print("Operational counts (not research metrics):")
for count in status_counts:
    print(count)
if MODE == "run":
    artifact_path = study.save_artifact(artifact, OUTPUT_DIR)
    print("Saved:", artifact_path)
print("All outputs are unscored. Preview fixtures are not research results.")

## Reproducibility and scope

Run every cell in order after changing the centralized controls. Inspect `artifact`, `rows`, `encoder_predictions`, raw prompts and `shown_order` as intermediate values. Use a fresh kernel for an independent run. Preview and reload never activate enabled model flags.

The [training appendix](train_dictabert.ipynb) describes encoder training separately. The [input preparation code](../src/hebrew_acronyms/data_processing/prepare_encoder_inputs.py) and [checkpoint implementation](../src/hebrew_acronyms/models/dictabert_cross_encoder/model.py) retain ownership of input and model identity. No research execution or scoring has been performed to prepare this notebook. AI assistance contributed the notebook glue and fixture checks; it does not constitute human annotation or scientific validation.